# In-class Activity #11 (August 6, 2026)

In [1]:
library(tidyverse)
library(moderndive)
library(car)

── Attaching core tidyverse packages ─────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ───────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: carData


Attaching package: ‘car’


The following object is masked from ‘package:dplyr’:

    recode


The following object is masked from ‘package:purrr’:

    some




In [2]:
wage_data <- read.table("data/wage.txt", header=TRUE)
wage_data$wage_categorical <- as.numeric(wage_data$wage > mean(wage_data$wage))
head(wage_data)

,education,south,sex,experience,union,wage,age,race,occupation,sector,marr,wage_categorical
,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>
1,8,0,1,21,0,5.10,35,2,6,1,1,0
2,9,0,1,42,0,4.95,57,3,6,1,1,0
3,12,0,0,1,0,6.67,19,3,6,1,0,0
4,12,0,0,4,0,4.00,22,3,6,0,0,0
5,12,0,0,17,0,7.50,35,3,6,0,1,0
6,13,0,0,9,1,13.07,28,3,6,0,0,1


### Part A

In [3]:
full_model <- glm(formula=wage_categorical ~ education + south + sex + 
                 experience + union + age + factor(race) + 
                 factor(occupation) + factor(sector) + marr, data= wage_data, family=binomial)
full_model


Call:  glm(formula = wage_categorical ~ education + south + sex + experience + 
    union + age + factor(race) + factor(occupation) + factor(sector) + 
    marr, family = binomial, data = wage_data)

Coefficients:
        (Intercept)            education                south  
            16.1581               3.9470              -0.3394  
                sex           experience                union  
            -0.8410               3.6514               1.2531  
                age        factor(race)2        factor(race)3  
            -3.6065              -0.1093               0.6449  
factor(occupation)2  factor(occupation)3  factor(occupation)4  
            -1.3137              -0.9538              -1.9541  
factor(occupation)5  factor(occupation)6      factor(sector)1  
            -0.6302              -1.3601               0.8922  
    factor(sector)2                 marr  
             1.0538               0.4006  

Degrees of Freedom: 533 Total (i.e. Null);  517 Residual
N

In [4]:
stepwise_model <- step(full_model, direction="backward")
summary(stepwise_model)

Start:  AIC=563.39
wage_categorical ~ education + south + sex + experience + union + 
    age + factor(race) + factor(occupation) + factor(sector) + 
    marr

                     Df Deviance    AIC
- south               1   531.23 563.23
<none>                    529.39 563.39
- age                 1   531.76 563.76
- experience          1   532.00 564.00
- marr                1   532.15 564.15
- factor(race)        2   534.26 564.26
- education           1   533.88 565.88
- factor(sector)      2   539.75 569.75
- factor(occupation)  5   548.75 572.75
- sex                 1   541.29 573.29
- union               1   548.93 580.93

Step:  AIC=563.23
wage_categorical ~ education + sex + experience + union + age + 
    factor(race) + factor(occupation) + factor(sector) + marr

                     Df Deviance    AIC
<none>                    531.23 563.23
- age                 1   533.70 563.70
- marr                1   533.90 563.90
- experience          1   533.95 563.95
- factor(race


Call:
glm(formula = wage_categorical ~ education + sex + experience + 
    union + age + factor(race) + factor(occupation) + factor(sector) + 
    marr, family = binomial, data = wage_data)

Coefficients:
                    Estimate Std. Error z value Pr(>|z|)    
(Intercept)          16.0311   803.1175   0.020 0.984074    
education             3.9721   133.8528   0.030 0.976326    
sex                  -0.8224     0.2449  -3.358 0.000784 ***
experience            3.6696   133.8528   0.027 0.978128    
union                 1.2885     0.2893   4.454 8.44e-06 ***
age                  -3.6241   133.8528  -0.027 0.978400    
factor(race)2        -0.1196     0.6613  -0.181 0.856433    
factor(race)3         0.6903     0.3499   1.973 0.048472 *  
factor(occupation)2  -1.3497     0.5129  -2.632 0.008497 ** 
factor(occupation)3  -0.9864     0.4182  -2.359 0.018341 *  
factor(occupation)4  -1.9570     0.4919  -3.979 6.93e-05 ***
factor(occupation)5  -0.6534     0.4078  -1.602 0.109128    
f

I performed stepwise selection in the backwards direction, meaning that we start with the full model and iteratively remove predictors.

The step() function selects all the predictors except for `south`. However, since the step() function selects values by using AIC values, so we must observe the p-values to determine if the variables selected are statistically significant.

At a 0.05 significance level, sex, union, race, occupation, and sector are significant. Note: for the categorical variables, the statistical significance identified is simply between at least one of the categories compared against the reference variable.

Therefore, although education, experience, and age were selected by the stepwise function, their p-values indicate that they are not statistically significant. However, their identically inflated standard errors indicate there may be considerable multicollinearity.

### Part B

In [5]:
anova(stepwise_model, full_model)

,Resid. Df,Resid. Dev,Df,Deviance,Pr(>Chi)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,518,531.2263,NA,NA,NA
2,517,529.3853,1,1.841034,0.1748298


After running the deviance test between the reduced and full model, we can see that the deviance drops by `1.84` when adding the south variable to the model. The null hypothesis is that the smaller model (stepwise_model) is sufficient, whereas the H_1 is that the full model is significantly better. Since the p-value is `0.174`, at a 0.05 significance level, we fail to reject the null hypothesis, therefore we can conclude that there is not enough evidence to support the addition of the `south` predictor.